# V-Max BC training on Colab

Runs `algorithm=bc` training from `as-fast-as-anyone` (V-Max fork) on a Colab GPU instead of the local GTX 1660 Super (6GB VRAM / 15GB RAM), using the hard/easy BC data pools.

**Before running this notebook**, upload the BC data pools to your Google Drive at:
```
MyDrive/vmax_workdir/data/shards/bc_pools/hard/   (local: data/shards/bc_pools/hard/, ~23,570 shards)
MyDrive/vmax_workdir/data/shards/bc_pools/easy/   (local: data/shards/bc_pools/easy/, ~35,355 shards)
```
You do **not** need the 179GB `data/train_91f/` - only these already-built shard pools (a few hundred MB) are used for training.

Runtime > Change runtime type > select a GPU (T4 is free-tier; Colab Pro gives A100/L4).

**Why Drive at all**: Colab sessions disconnect (idle timeout / max runtime). Checkpoints are written straight to Drive (via a symlink), and BC training now supports full resume - if the session dies, just re-run the training cell with the same `name_run` and it picks up from the last checkpoint instead of restarting.

In [ ]:
!nvidia-smi

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_WORKDIR = "/content/drive/MyDrive/vmax_workdir"
import os
assert os.path.isdir(f"{DRIVE_WORKDIR}/data/shards/bc_pools/hard"), (
    f"Missing {DRIVE_WORKDIR}/data/shards/bc_pools/hard - upload the bc_pools folders to Drive first (see markdown above)."
)
assert os.path.isdir(f"{DRIVE_WORKDIR}/data/shards/bc_pools/easy"), (
    f"Missing {DRIVE_WORKDIR}/data/shards/bc_pools/easy - upload the bc_pools folders to Drive first (see markdown above)."
)
print("Drive OK, bc_pools found.")

## 2. Clone the repo and set up the environment (uv, pinned by uv.lock)

In [ ]:
%cd /content
!rm -rf as-fast-as-anyone
!git clone https://github.com/gm2256/as-fast-as-anyone.git
%cd /content/as-fast-as-anyone/V-Max

!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"{os.path.expanduser('~')}/.local/bin:" + os.environ["PATH"]
!uv --version

In [ ]:
# Installs its own Python 3.12 (per .python-version) regardless of Colab's system Python,
# and resolves the exact versions pinned in uv.lock (same env as the local machine).
!uv sync

## 3. Wire up data (local disk, fast) and checkpoints (Drive, persistent)

Data is copied to local Colab disk once (repeated shuffled reads during training are much faster off local SSD than off the Drive FUSE mount). `runs/` is symlinked into Drive so checkpoints/logs survive a disconnect.

In [ ]:
!mkdir -p /content/data/shards/bc_pools
!cp -r "$DRIVE_WORKDIR/data/shards/bc_pools/hard" /content/data/shards/bc_pools/hard
!cp -r "$DRIVE_WORKDIR/data/shards/bc_pools/easy" /content/data/shards/bc_pools/easy
!ls /content/data/shards/bc_pools/hard | wc -l
!ls /content/data/shards/bc_pools/easy | wc -l

In [ ]:
import os
os.makedirs(f"{DRIVE_WORKDIR}/runs", exist_ok=True)
!rm -rf /content/as-fast-as-anyone/V-Max/runs
!ln -s "$DRIVE_WORKDIR/runs" /content/as-fast-as-anyone/V-Max/runs
!ls -la /content/as-fast-as-anyone/V-Max/runs

## 4. Train

`@23570` / `@35355` must match however many shard files you actually uploaded for hard/easy - check the `wc -l` counts printed above and edit if they differ.

`total_timesteps=5_000_000` is roughly one pass over the combined hard+easy pool (~59k scenarios x 80 steps). Bump it up (e.g. `20_000_000`, the framework's own default scale) once you've confirmed `train/imitation_loss` in TensorBoard is still trending down at 5M and want to keep going.

**If the session disconnects mid-run**: just re-run this cell unchanged. `algorithm.resume=true` (default) picks up from `runs/<name_run>/model/train_state_latest.pkl` on Drive.

In [ ]:
%cd /content/as-fast-as-anyone/V-Max
!uv run python vmax/scripts/training/train.py \
  algorithm=bc network/encoder=lq \
  total_timesteps=5_000_000 num_envs=4 num_episode_per_epoch=1 \
  algorithm.buffer_size=20000 \
  waymo_dataset=true \
  'mixture_datasets=[{path: /content/data/shards/bc_pools/hard/hard.tfrecord@23570, weight: 0.3}, {path: /content/data/shards/bc_pools/easy/easy.tfrecord@35355, weight: 0.7}]' \
  name_run=colab_bc_run1 log_freq=50 save_freq=1500

## 5. Watch training in TensorBoard (optional, run in a separate cell while training runs)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/as-fast-as-anyone/V-Max/runs

## 6. After training: sweep checkpoints on the fixed held-out set

Upload `data/eval/val_sample_shards_hanam/` (the fixed 300-scenario validation set) to
`MyDrive/vmax_workdir/data/eval/val_sample_shards_hanam/` first.

In [ ]:
!mkdir -p /content/data/eval
!cp -r "$DRIVE_WORKDIR/data/eval/val_sample_shards_hanam" /content/data/eval/val_sample_shards_hanam

%cd /content/as-fast-as-anyone/V-Max
!uv run python scripts/evaluate_checkpoints.py \
  --name_run colab_bc_run1 \
  --path_dataset /content/data/eval/val_sample_shards_hanam/val_sample_shards_hanam.tfrecord@300 \
  --waymo_dataset true --batch_size 4